# DataLoad de testes  Argos

Massa fictícia para uma cópia limpa do banco

## 1. Conexão

In [ ]:
import os
import uuid
from datetime import date, time, timedelta

import psycopg2
from dotenv import load_dotenv

load_dotenv()

conexao = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT"),
)
cursor = conexao.cursor()
hoje = date.today()

print("Conexão realizada com sucesso.")

## 2. Dados fixos já existentes

In [ ]:
cursor.execute("SELECT code, id FROM permission")
permissoes = dict(cursor.fetchall())

cursor.execute("SELECT name, id FROM therapeutic_category")
categorias_terapeuticas = dict(cursor.fetchall())

cursor.execute("SELECT id FROM app_user WHERE cpf = '00000000000'")
resultado_rpa = cursor.fetchone()

if "PECUARISTA" not in permissoes:
    raise RuntimeError("Permissão PECUARISTA não encontrada")

if resultado_rpa is None:
    raise RuntimeError("Usuário técnico da RPA não encontrado")

permission_id = permissoes["PECUARISTA"]
usuario_rpa = resultado_rpa[0]

print("Permissões, categorias e usuário da RPA encontrados.")

## 3. Usuários e propriedades

In [ ]:
nomes_usuarios = [
    "Ana Silva", "Bruno Costa", "Carlos Eduardo", "Daniela Rocha", "Eduardo Lima",
    "Fernanda Souza", "Gabriel Alves", "Helena Martins", "Igor Ferreira",
    "Juliana Paes", "Lucas Mendes", "Mariana Oliveira", "Nicolas Santos",
    "Patricia Ribeiro", "Rafael Barbosa",
]

usuarios = []

for numero in range(15):
    usuario = (
        uuid.uuid4(),
        permission_id,
        nomes_usuarios[numero],
        "123456789" + str(numero + 1).zfill(2),
        "usuario" + str(numero + 1).zfill(2) + "@argos.test",
        False,
        True,
    )
    usuarios.append(usuario)

cursor.executemany(
    """
    INSERT INTO app_user
        (id, permission_id, name, cpf, email, first_login, active)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """,
    usuarios,
)

nomes_propriedades = [
    "Fazenda Sol Nascente", "Sítio das Águas", "Fazenda Santa Maria",
    "Estância Boi Gordo", "Fazenda Monte Verde", "Sítio Recanto Feliz",
    "Fazenda Palmeiras", "Fazenda Planalto", "Sítio Boa Vista",
    "Fazenda Vale Verde", "Estância Querência", "Fazenda São João",
    "Sítio Alvorada", "Fazenda Ouro Branco", "Fazenda Terra Prometida",
]
cidades = ["Ribeirão Preto", "Barretos", "Sertãozinho", "Cravinhos", "Bebedouro"]
ruas = ["Rua Central", "Estrada Rural", "Avenida Industrial", "Rodovia Municipal", "Estrada da Boiada"]

propriedades = []

for indice in range(15):
    propriedade = (
        uuid.uuid4(),
        usuarios[indice][0],
        nomes_propriedades[indice],
        "123456780001" + str(indice + 1).zfill(2),
        ruas[indice % len(ruas)],
        str(100 + indice),
        "Zona Rural",
        "14" + str(indice + 1).zfill(6),
        cidades[indice % len(cidades)],
        "SP",
        150 + indice * 35,
        "16333344" + str(indice + 1).zfill(2),
        True,
    )
    propriedades.append(propriedade)

cursor.executemany(
    """
    INSERT INTO property
        (id, user_id, name, cnpj, street, number, complement, zip_code,
         city, state, area_hectares, phone, active)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    propriedades,
)

vinculos_propriedade = []

for indice in range(15):
    vinculo = (usuarios[indice][0], propriedades[indice][0])
    vinculos_propriedade.append(vinculo)

cursor.executemany(
    "INSERT INTO user_property (user_id, property_id) VALUES (%s, %s)",
    vinculos_propriedade,
)

print(f"{len(usuarios)} usuários e {len(propriedades)} propriedades inseridos.")

## 4. Rebanhos e lotes

In [ ]:
nomes_rebanhos = [
    "Nelore Elite A", "Angus Premium", "Gyr Corte", "Brahman Comercial",
    "Brangus Engorda", "Senepol Puro", "Girolando Recria", "Nelore Mocho",
    "Hereford Sul", "Wagyu Seleção", "Tabapuã Nelore", "Holandês Machos Corte",
    "Guzerá Rústico", "Canchim Confinamento", "Santa Gertrudis",
]
racas = [
    "Nelore", "Angus", "Gyr", "Brahman", "Brangus", "Senepol", "Girolando",
    "Nelore", "Hereford", "Wagyu", "Tabapuã", "Holandês", "Guzerá",
    "Canchim", "Santa Gertrudis",
]

rebanhos = []

for indice in range(30):
    nome_rebanho = nomes_rebanhos[indice % 15]
    if indice >= 15:
        nome_rebanho = nome_rebanho + " II"

    rebanho = (
        uuid.uuid4(),
        propriedades[indice % len(propriedades)][0],
        nome_rebanho,
        racas[indice % len(racas)],
        True,
    )
    rebanhos.append(rebanho)

cursor.executemany(
    """
    INSERT INTO herd (id, property_id, herd_name, breed, active)
    VALUES (%s, %s, %s, %s, %s)
    """,
    rebanhos,
)

categorias_lote = ["CRIA", "RECRIA", "ENGORDA"]

lotes = []

for indice in range(100):
    lote = (
        uuid.uuid4(),
        rebanhos[indice % len(rebanhos)][0],
        "LOTE-" + str(indice + 1).zfill(3),
        categorias_lote[indice % len(categorias_lote)],
        50,
        hoje - timedelta(days=indice % 120),
        True,
    )
    lotes.append(lote)

cursor.executemany(
    """
    INSERT INTO lot
        (id, herd_id, lot_code, category, head_count, open_date, active)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """,
    lotes,
)

print(f"{len(rebanhos)} rebanhos e {len(lotes)} lotes principais inseridos.")

## 5. Lotes e animais de exceção

In [ ]:
motivos_excecao = [
    "Tratamento de pneumonia e isolamento",
    "Tratamento de miíase no casco",
    "Fotossensibilização por intoxicação alimentar",
    "Reação vacinal intensa",
    "Ferimento por arame",
    "Baixa condição corporal",
    "Queratoconjuntivite",
    "Tristeza Parasitária Bovina",
    "Abscesso sob investigação",
    "Diarreia infecciosa",
]

lotes_excecao = []

for indice in range(20):
    lote_excecao = (
        uuid.uuid4(),
        lotes[indice][1],
        lotes[indice][0],
        "EXC-" + str(indice + 1).zfill(3),
        lotes[indice][3],
        True,
        motivos_excecao[indice % len(motivos_excecao)],
        0,
        False,
        True,
    )
    lotes_excecao.append(lote_excecao)

cursor.executemany(
    """
    INSERT INTO lot
        (id, herd_id, parent_lot_id, lot_code, category, is_exception_lot,
         exception_reason, head_count, fit_for_slaughter, active)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    lotes_excecao,
)

animais = []

for indice in range(20):
    animal = (
        uuid.uuid4(),
        lotes_excecao[indice][0],
        lotes[indice][0],
        "BR-" + str(indice + 1).zfill(3),
        "BOI",
        hoje - timedelta(days=800 + indice * 15),
        350 + indice * 7,
        motivos_excecao[indice % len(motivos_excecao)],
        "ATIVO",
    )
    animais.append(animal)

cursor.executemany(
    """
    INSERT INTO animal_exception
        (id, lot_id, origin_lot_id, ear_tag, category, birth_date,
         current_weight, exception_reason, status)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    animais,
)

historicos = []

for indice in range(20):
    historico = (uuid.uuid4(), animais[indice][0], lotes_excecao[indice][0], hoje, "EXCECAO")
    historicos.append(historico)

cursor.executemany(
    """
    INSERT INTO animal_lot_history
        (id, animal_exception_id, lot_id, entry_date, reason)
    VALUES (%s, %s, %s, %s, %s)
    """,
    historicos,
)

print(f"{len(lotes_excecao)} lotes de exceção e {len(animais)} animais inseridos.")

## 6. Fornecedores e medicamentos

In [ ]:
nomes_fornecedores = [
    "AgroPharma Brasil", "VetMedic Distribuidora", "Laboratório AgroSaúde",
    "BioVet Insumos", "RuralVet Produtos", "PharmaGado Ltda",
    "NutriVet Saúde Animal", "Central Vet Insumos", "VetLife Soluções",
    "AgroMedic Comércio", "ProCampo Medicamentos", "GadoForte Insumos",
    "BovinoVet Distribuição", "AgroSanidade Brasil", "MegaVet Soluções",
]

fornecedores = []

for indice in range(15):
    fornecedor = (
        uuid.uuid4(),
        nomes_fornecedores[indice],
        "111111110001" + str(indice + 1).zfill(2),
        "contato" + str(indice + 1).zfill(2) + "@fornecedor.test",
        "163901" + str(indice + 1).zfill(4),
        "Distrito Industrial, bloco " + str(indice + 1),
        True,
    )
    fornecedores.append(fornecedor)

cursor.executemany(
    """
    INSERT INTO supplier
        (id, company_name, cnpj, email, phone, address, active)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """,
    fornecedores,
)

catalogo_medicamentos = [
    ("Ivervet 1%", "Ivermectina", "Controle de parasitas internos e externos", 35, "Antiparasitário", 10, "SUBCUTANEA"),
    ("Bovivet Penicilina", "Penicilina G Procaína", "Tratamento de infecções bacterianas", 30, "Antibiótico", 15, "INTRAMUSCULAR"),
    ("DexaGado Forte", "Dexametasona", "Anti-inflamatório esteroidal", 15, "Anti-inflamatório", 5, "INTRAMUSCULAR"),
    ("Carrapatox Spray", "Cipermetrina", "Controle de carrapatos e moscas", 14, "Ectoparasiticida", 20, "TOPICA"),
    ("Oxivet LA", "Oxitetraciclina", "Antibiótico de amplo espectro", 28, "Antibiótico", 10, "SUBCUTANEA"),
    ("DoraVet 3.15%", "Doramectina", "Antiparasitário de longa ação", 42, "Antiparasitário", 10, "SUBCUTANEA"),
    ("Flunixin Campo", "Flunixin Meglumine", "Anti-inflamatório não esteroidal", 10, "Anti-inflamatório", 8, "INTRAMUSCULAR"),
    ("Ceftiofur Max", "Ceftiofur", "Infecções respiratórias bovinas", 4, "Antibiótico", 5, "INTRAMUSCULAR"),
    ("SupleVet B12", "Vitamina B12", "Suplementação vitamínica", 0, "Vitamínico", 10, "INTRAMUSCULAR"),
    ("Complexo B Injetável", "Complexo B", "Suplemento nutricional", 0, "Vitamínico", 5, "INTRAMUSCULAR"),
    ("Vacina RaivaVet", "Vírus Inativado da Raiva", "Imunização contra raiva", 0, "Vacina", 2, "SUBCUTANEA"),
    ("Vacina AftosaShield", "Antígeno Inativado Aftosa", "Imunização contra febre aftosa", 0, "Vacina", 5, "SUBCUTANEA"),
    ("Cefalexina Injetável", "Cefalexina", "Tratamento de infecções em geral", 7, "Antibiótico", 10, "INTRAMUSCULAR"),
    ("Albendazol Pour-On", "Albendazol", "Vermífugo de amplo espectro", 14, "Antiparasitário", 15, "TOPICA"),
    ("CalciumVet Injectable", "Gluconato de Cálcio", "Tratamento de hipocalcemia", 0, "Suplemento Mineral", 50, "INTRAVENOSA"),
]

medicamentos = []

for indice in range(15):
    dados = catalogo_medicamentos[indice]
    nome = dados[0]
    principio = dados[1]
    indicacao = dados[2]
    carencia = dados[3]
    categoria = dados[4]
    dose = dados[5]
    via = dados[6]

    medicamento = (
        uuid.uuid4(),
        fornecedores[indice][0],
        categorias_terapeuticas[categoria],
        nome,
        principio,
        indicacao,
        "Medicamento fictício para homologação",
        categoria == "Antibiótico",
        carencia,
        dose,
        "ml",
        via,
        True,
    )
    medicamentos.append(medicamento)

cursor.executemany(
    """
    INSERT INTO medicine
        (id, supplier_id, therapeutic_category_id, brand_name,
         active_ingredient, indication, description, needs_prescription,
         meat_withdrawal_days, dose, unit, route, active)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    medicamentos,
)

print(f"{len(fornecedores)} fornecedores e {len(medicamentos)} medicamentos inseridos.")

## 7. Lotes de medicamento e estoque

In [ ]:
lotes_medicamento = []

for indice in range(45):
    lote_medicamento = (
        uuid.uuid4(),
        medicamentos[indice % len(medicamentos)][0],
        "LOTE-MED-" + str(indice + 1).zfill(3),
        hoje - timedelta(days=180),
        hoje + timedelta(days=540),
        hoje - timedelta(days=90),
        5000,
        500,
        7000,
        "Galpão Central - Prateleira " + str((indice % 10) + 1),
        "DISPONIVEL",
    )
    lotes_medicamento.append(lote_medicamento)

cursor.executemany(
    """
    INSERT INTO medicine_batch
        (id, medicine_id, batch_number, manufacture_date, expiry_date,
         delivery_date, initial_quantity, min_stock, max_stock,
         storage_place, status)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    lotes_medicamento,
)

movimentos_iniciais = []

for lote_medicamento in lotes_medicamento:
    movimento = (uuid.uuid4(), lote_medicamento[0], usuario_rpa, "E", 5000, "CARGA_INICIAL")
    movimentos_iniciais.append(movimento)

cursor.executemany(
    """
    INSERT INTO stock_movement
        (id, medicine_batch_id, user_id, type, quantity, reason)
    VALUES (%s, %s, %s, %s, %s, %s)
    """,
    movimentos_iniciais,
)

print(f"{len(lotes_medicamento)} lotes de medicamento e entradas de estoque inseridos.")

## 8. Aplicações

In [ ]:
objetivos = [
    "Vermifugação de rotina", "Tratamento preventivo", "Redução de inflamação",
    "Controle de carrapatos", "Tratamento antibiótico", "Imunização vacinal",
]

aplicacoes_lote = []

for indice in range(260):
    aplicacao = (
        uuid.uuid4(),
        lotes[indice % len(lotes)][0],
        lotes_medicamento[indice % len(lotes_medicamento)][0],
        usuarios[indice % len(usuarios)][0],
        hoje - timedelta(days=indice % 30),
        time(7 + indice % 10, (indice * 5) % 60),
        2,
        "ml",
        catalogo_medicamentos[indice % len(catalogo_medicamentos)][6],
        5,
        "Pescoço",
        "Curral de manejo",
        objetivos[indice % len(objetivos)],
        "Aplicação coletiva de teste " + str(indice + 1).zfill(3),
    )
    aplicacoes_lote.append(aplicacao)

cursor.executemany(
    """
    INSERT INTO application
        (id, lot_id, medicine_batch_id, user_id, application_date,
         application_time, dose, unit, route, head_count_applied,
         applied_limb, applied_site, goal, notes)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    aplicacoes_lote,
)

aplicacoes_individuais = []

for indice in range(20):
    aplicacao = (
        uuid.uuid4(),
        lotes_excecao[indice][0],
        lotes_medicamento[indice][0],
        usuarios[indice % len(usuarios)][0],
        hoje - timedelta(days=indice % 15),
        time(9, 0),
        2,
        "ml",
        "INTRAMUSCULAR",
        1,
        "Pescoço",
        "Enfermaria",
        "Tratamento individual",
        "Aplicação individual de teste " + str(indice + 1).zfill(2),
    )
    aplicacoes_individuais.append(aplicacao)

cursor.executemany(
    """
    INSERT INTO application
        (id, lot_id, medicine_batch_id, user_id, application_date,
         application_time, dose, unit, route, head_count_applied,
         applied_limb, applied_site, goal, notes)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    aplicacoes_individuais,
)

vinculos_aplicacao_animal = []

for indice in range(20):
    vinculo = (uuid.uuid4(), aplicacoes_individuais[indice][0], animais[indice][0], 2, animais[indice][6])
    vinculos_aplicacao_animal.append(vinculo)

cursor.executemany(
    """
    INSERT INTO application_animal
        (id, application_id, animal_exception_id, dose_given, weight_at_time)
    VALUES (%s, %s, %s, %s, %s)
    """,
    vinculos_aplicacao_animal,
)

print(f"{len(aplicacoes_lote) + len(aplicacoes_individuais)} aplicações inseridas.")

## 9. Ocorrências

In [ ]:
tipos_ocorrencia = ["DOENCA", "MORTE", "REACAO_ADVERSA", "FERIMENTO"]

ocorrencias = []
for indice in range(30):
    data_ocorrencia = hoje - timedelta(days=indice % 25)
    resolvida = indice % 2 == 0
    ocorrencias.append(
        (
            uuid.uuid4(),
            animais[indice % len(animais)][0],
            usuarios[indice % len(usuarios)][0],
            tipos_ocorrencia[indice % len(tipos_ocorrencia)],
            data_ocorrencia,
            f"Ocorrência veterinária de teste {indice + 1:03d}",
            "Diagnóstico fictício para homologação",
            resolvida,
            data_ocorrencia + timedelta(days=2) if resolvida else None,
        )
    )

cursor.executemany(
    """
    INSERT INTO occurrence
        (id, animal_exception_id, user_id, type, occurrence_date,
         description, diagnosis, resolved, resolved_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    ocorrencias,
)

print(f"{len(ocorrencias)} ocorrências inseridas.")

## 10. Confirmar carga e conferir o total

In [ ]:
conexao.commit()

contagens = {
    "app_user": len(usuarios),
    "property": len(propriedades),
    "user_property": len(vinculos_propriedade),
    "herd": len(rebanhos),
    "lot": len(lotes) + len(lotes_excecao),
    "animal_exception": len(animais),
    "animal_lot_history": len(historicos),
    "supplier": len(fornecedores),
    "medicine": len(medicamentos),
    "medicine_batch": len(lotes_medicamento),
    "stock_movement inicial": len(movimentos_iniciais),
    "application": len(aplicacoes_lote) + len(aplicacoes_individuais),
    "application_animal": len(vinculos_aplicacao_animal),
    "occurrence": len(ocorrencias),
}

for tabela, quantidade in contagens.items():
    print(f"{tabela:<28} {quantidade:>4}")

print(f"{'TOTAL ÚTIL':<28} {sum(contagens.values()):>4}")
print("Alertas, auditorias e saídas automáticas não foram contados.")

cursor.close()
conexao.close()